# Conditional Columns

Sometimes a value in a new column depends on a condition. In this notebook, we will use `F.when()` and `otherwise()` to assign useful categories to each employee.

---
## Learning objectives

By the end of this notebook, you will be able to:

- create a two-outcome conditional column with `F.when()` and `otherwise()`;
- create a column with multiple conditional outcomes; and
- order conditions so that the intended category is assigned.

---
## Set up the example data

This notebook is self-contained. It recreates the employee DataFrame from the previous lesson; in Microsoft Fabric, the `spark` session is already available.

In [ ]:
from pyspark.sql import functions as F

employee_rows = [
    (101, "Aisha Khan", "Engineering", 2021, 72000),
    (102, "Ben Carter", "Finance", 2019, 68000),
    (103, "Chloe Martin", "Engineering", 2023, 61000),
    (104, "Daniel Wong", "People", 2020, 59000),
    (105, "Elena Rossi", "Finance", 2022, 65000),
]

employee_schema = """
    employee_id INT,
    name STRING,
    department STRING,
    start_year INT,
    salary INT
"""

employees = spark.createDataFrame(employee_rows, schema=employee_schema)
employees.show()

---
## Recap: create a new DataFrame

`withColumn()` returns a new DataFrame with a new or replaced column. `F.col()` supplies the value from a column for each row; `F.when()` decides which new value to use. The original `employees` DataFrame remains unchanged.

---
## Creating a new column with a two-outcome conditional column

This expression creates `salary_category`. Employees with a salary of at least 70,000 receive `High`; everyone else receives `Standard`.

In [ ]:
employees_with_salary_category = employees.withColumn(
    "salary_category",
    F.when(F.col("salary") >= 70000, "High").otherwise("Standard"),
)

employees_with_salary_category.show()

### Why use `otherwise()`?

`otherwise()` supplies the value for every row that did not match a `when()` condition. If it is omitted, unmatched rows receive `NULL`.



### Equivalent SQL: `CASE` statement

`F.when(...).otherwise(...)` is the PySpark DataFrame equivalent of a SQL `CASE WHEN ... ELSE ... END` expression. The code above is equivalent to:

```sql
CASE
    WHEN salary >= 70000 THEN 'High'
    ELSE 'Standard'
END AS salary_category
```

---
## Multiple conditional outcomes

Chain additional `when()` calls to create more than two categories. Here we create three salary bands.

In [ ]:
employees_with_salary_band = employees.withColumn(
    "salary_band",
    F.when(F.col("salary") >= 70000, "High")
    .when(F.col("salary") >= 60000, "Medium")
    .otherwise("Foundation"),
)

employees_with_salary_band.show()

### Condition order matters

Spark checks chained `when()` conditions from top to bottom and uses the first match. The `>= 70000` condition must come before `>= 60000`; otherwise, a salary of 72,000 would be labelled `Medium` before Spark reaches the `High` condition.

---
## Your turn

**Exercise 1:** Create a DataFrame named `employees_with_status` with a new `employment_status` column. Employees with `start_year` less than or equal to 2020 should be `Established`; all other employees should be `Recent hire`. Preview the result with `show()`.

In [ ]:
# Write your solution here.

**Exercise 2:** Create a DataFrame named `employees_with_salary_band` with a three-level `salary_band` column: `High` for salary at least 70,000, `Medium` for salary at least 60,000, and `Foundation` otherwise. Preview the result with `show()`.

In [ ]:
# Write your solution here.

---
## Next lesson

Next, we will identify, replace, and remove missing values before working with larger summaries and joins.